#transformations

## customers

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df_cust= spark.read.csv("abfss://bronze@oliststoragedatalake.dfs.core.windows.net/customers", header=True,inferSchema=True)

In [0]:



df_customer=df_cust.select(col("customer_id").cast("String"),
                       col("customer_unique_id"),col("customer_zip_code_prefix").cast("Integer"),
                       upper(col("customer_city")).cast("String").alias("customer_city"),upper(col("customer_state")).cast("String").alias("customer_state"))
df_customer.show(10,False)


In [0]:
df_customer.write.mode("overwrite").format("delta").save("abfss://silver@oliststoragedatalake.dfs.core.windows.net/customers")

## orderitems

In [0]:
df_oi = spark.read.csv("abfss://bronze@oliststoragedatalake.dfs.core.windows.net/orderitems", header=True,inferSchema=True)

In [0]:
df_orderitems=df_oi.withColumn("shipping_limit_date", to_date(col("shipping_limit_date"),"yyyy-MM-dd"))
df_orderitems.show(10,False)


In [0]:
df_orderitems.write.mode("overwrite").format("delta").save("abfss://silver@oliststoragedatalake.dfs.core.windows.net/orderitems")

## orders

In [0]:
df_or= spark.read.csv("abfss://bronze@oliststoragedatalake.dfs.core.windows.net/orders", header=True,inferSchema=True)


In [0]:
df_or.show(1,False)
#df_or.select(col("order_status")).distinct()
df_or.join(df_customer,on="customer_id",how="inner")
df_orders=df_or.select(col("order_id"),col("customer_id"),col("order_status"),col("order_purchase_timestamp").cast("timestamp"),col("order_approved_at").cast("timestamp"),col("order_delivered_carrier_date").cast("date"),col("order_delivered_customer_date").cast("date"),col("order_estimated_delivery_date").cast("date"))
df_orders.show(10,False)

In [0]:
df_orders.write.mode("overwrite").format("delta").save("abfss://silver@oliststoragedatalake.dfs.core.windows.net/orders")

## products

In [0]:
df_pd=spark.read.csv("abfss://bronze@oliststoragedatalake.dfs.core.windows.net/products", header=True,inferSchema=True)


In [0]:
df_pd.show(10,False)

In [0]:
df_pd.write.mode("overwrite").format("delta").save("abfss://silver@oliststoragedatalake.dfs.core.windows.net/products")

## sellers

In [0]:
df_sell=spark.read.csv("abfss://bronze@oliststoragedatalake.dfs.core.windows.net/sellers", header=True,inferSchema=True)
#df_sell.show(10,False)
df_sell.withColumn("city",upper(split("seller_city"," ")[0]))
df_sell.createOrReplaceTempView("sell")
df_sellers=spark.sql("select seller_id,cast(seller_zip_code_prefix as int) seller_zip_code_prefix,upper(seller_city)seller_city,seller_state from sell")
df_sellers.show(10,False)


In [0]:
df_sellers.write.mode("overwrite").format("delta").save("abfss://silver@oliststoragedatalake.dfs.core.windows.net/sellers")


## order payments

In [0]:
df_op= spark.read.csv("abfss://bronze@oliststoragedatalake.dfs.core.windows.net/orderpayments", header=True,inferSchema=True)

In [0]:
#df_op.select("order_id").distinct().count()
df_orderpayments=df_op.join(df_or,on="order_id",how="inner")
df_orderpayments.show(10,False)

In [0]:
df_orderpayments.write.mode("overwrite").format("delta").save("abfss://silver@oliststoragedatalake.dfs.core.windows.net/orderpayments")